# Geometric similarity & distance metrics — a retrieval lab

**You implement. This notebook scaffolds.**

Every function whose body is `raise NotImplementedError` is yours to write. The
docstring gives you the signature, the shapes, and the formula in math notation —
never the code. The verification cells below each stub are mine: run them, don't
edit them, and let them tell you whether you're right.

**The task, everywhere:** for each sample, retrieve its `K = 10` nearest
neighbours *excluding itself*, then score

> `precision@10` = fraction of those 10 that share the query's true label

and report the mean and std across all queries.

**Two datasets, chosen to disagree:** 20newsgroups TF-IDF (cosine should win) and
`load_wine` standardised (Euclidean should win).

**Per-stage convention:** a markdown cell explaining what you're about to build and
why → a code cell with the stub → a code cell that verifies it and prints the result.

We build one stage at a time. Finish a stage, then ask me for the next one.

---
### Stages
| # | | status |
|---|---|---|
| 0 | retrieval harness | ← you are here |
| 1 | TF-IDF + cosine baseline | |
| 2 | TF-IDF + Euclidean, unnormalised | |
| 3 | L2-normalise → Euclidean == cosine | |
| 4 | Minkowski sweep over p | |
| 5 | weighted cosine, w = idf² | |
| 6 | soft cosine (toy + GloVe) | |
| 7 | wine: cosine / Euclidean / Mahalanobis | |
| 8 | the four distance→similarity conversions | |


## Stage 0 — the retrieval harness

Before any metric exists, you need the thing that turns a score matrix into a
number. Both stubs below are used by every single stage after this, so a bug here
will look like a metric result for the rest of the notebook.

Two decisions worth thinking about before you write:

- **A query must not retrieve itself.** `scores[i, i]` is `1.0` for cosine and
  `0.0` for any distance, so `i` is always its own best match. If you forget this,
  every precision score inherits one free correct hit — `precision@10` gets a
  floor of `0.1` and, at `k=1`, is just `1.0` everywhere.
- **One function, both directions.** Similarities rank high-to-low, distances
  low-to-high. Rather than write two functions, take a flag — you'll pass
  distances in stages 2, 3, 4 and 7, and similarities in 1, 5 and 6.


In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.spatial.distance import cdist
from sklearn.datasets import fetch_20newsgroups, load_wine
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.preprocessing import StandardScaler, normalize

K = 10
CATEGORIES = ["alt.atheism", "comp.graphics", "sci.space", "talk.politics.mideast"]
SEED = 0
np.set_printoptions(precision=4, suppress=True)

# Every stage appends here; the final summary table just reads it.
RESULTS = {}  # (metric, dataset) -> (mean, std)
print("ready")

In [ ]:
def top_k_neighbours(scores, k, higher_is_better):
    """Return each row's k best neighbours, EXCLUDING the query itself.

    Parameters
    ----------
    scores : ndarray (n, n)
        Pairwise scores. Similarities (higher = closer) or distances
        (lower = closer); `higher_is_better` says which.
    k : int
        How many neighbours to return.
    higher_is_better : bool
        True for similarity matrices, False for distance matrices.

    Returns
    -------
    ndarray (n, k) of int
        Row i lists query i's neighbours, best first. i must never appear in row i.

    Notes
    -----
    One clean trick does the self-exclusion without special-casing: make the
    diagonal the worst possible value before sorting, so it always sorts last.
    """
    raise NotImplementedError


def precision_at_k(neighbour_idx, labels):
    """Fraction of each query's retrieved neighbours sharing its true label.

        precision@k(i) = (1/k) * SUM_{j in N_k(i)} 1[ y_j == y_i ]

    Parameters
    ----------
    neighbour_idx : ndarray (n, k) of int
        Output of `top_k_neighbours`.
    labels : ndarray (n,)
        True label per sample.

    Returns
    -------
    ndarray (n,) of float
        Per-query precision, each in [0, 1]. Report `.mean()` and `.std()` of this.
        Keep it per-query rather than pre-averaged: the std across queries is the
        noise floor you'll need in stage 4 to judge whether a difference is real.
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit)
# Four points on a line, worked out by hand so the harness is proven before any
# real data touches it.
#
#     position   0.0   1.1   2.3   3.7
#     label       0     1     0     1        <- deliberately interleaved
#
# k=1, self excluded : every point's nearest neighbour carries the OTHER label
#                      -> per-query [0, 0, 0, 0], mean 0.0
#                      -> if you forgot to exclude self you'd get 1.0 instead
# k=2, self excluded : per-query [0.5, 0.0, 0.0, 0.5] -> mean 0.25, std 0.25

pos = np.array([0.0, 1.1, 2.3, 3.7])
lab = np.array([0, 1, 0, 1])
D = np.abs(pos[:, None] - pos[None, :])

p1 = precision_at_k(top_k_neighbours(D, 1, higher_is_better=False), lab)
assert p1.mean() == 0.0, f"k=1 gave {p1.mean()!r}; 1.0 means the query retrieved itself"

p2 = precision_at_k(top_k_neighbours(D, 2, higher_is_better=False), lab)
assert np.allclose(p2, [0.5, 0.0, 0.0, 0.5]), f"k=2 per-query was {p2}"
assert np.allclose([p2.mean(), p2.std()], [0.25, 0.25])

# -D is the same ranking expressed as a similarity; the flag must absorb the flip.
assert np.array_equal(top_k_neighbours(-D, 2, True), top_k_neighbours(D, 2, False)), \
    "higher_is_better isn't flipping the order"

# shape and dtype contract
nb_ = top_k_neighbours(D, 2, False)
assert nb_.shape == (4, 2) and np.issubdtype(nb_.dtype, np.integer)

print("harness OK — self-exclusion, both directions, shapes, and precision all check out")

## Stage 1 — TF-IDF + cosine baseline

The number every later stage is measured against.

Cosine divides out vector length, so a 50-word post and a 2000-word post on the
same topic can still be nearest neighbours. On text that is exactly what you want:
length is a property of the author, not of the topic. Stage 2 removes that
division and you'll watch the result collapse.

You'll write cosine from scratch **and** call sklearn's, then assert they agree —
the point being that the formula and the library call are the same object.


In [ ]:
# DATA LOADING (mine). Two choices here are deliberate and worth reading:
#
# 1. remove=('headers','footers','quotes'): the headers contain the newsgroup
#    name. Leave them in and you are not measuring a metric, you are measuring a
#    label leak.
# 2. norm=None: TfidfVectorizer L2-normalises by DEFAULT. If we let it, stages
#    1-3 would agree before you'd had a chance to see *why* they agree. The
#    normalising is yours to do in stage 3.

news = fetch_20newsgroups(subset="train", categories=CATEGORIES,
                          remove=("headers", "footers", "quotes"),
                          shuffle=True, random_state=SEED)
vect = TfidfVectorizer(norm=None, min_df=2, max_features=5000, stop_words="english")
X_text = vect.fit_transform(news.data)
y_text = news.target
docs = list(news.data)

print(f"{X_text.shape[0]} documents x {X_text.shape[1]} terms")
print(f"categories: {news.target_names}")
print(f"density: {X_text.nnz / (X_text.shape[0] * X_text.shape[1]):.3%} non-zero")

In [ ]:
# TRAP 2, planted on purpose — zero vectors.
#
# Stripping headers/footers/quotes empties some posts completely. A zero vector
# has no direction, so cosine is 0/0 = nan, and ONE nan silently poisons every
# argsort that touches it. sklearn hides this by returning 0.0 instead, which is
# arguably worse: the document is simply never retrieved and nothing warns you.

norms = np.sqrt(np.asarray(X_text.multiply(X_text).sum(axis=1))).ravel()
n_zero = int((norms == 0).sum())
print(f"{n_zero} of {len(norms)} documents have ||x|| == 0")
print(f"non-zero norms range {norms[norms > 0].min():.2f} .. {norms.max():.2f}")
print()
print("Handle this before you trust any number below. Defensible options:")
print("  (a) drop those rows from X_text AND y_text AND docs — keep them aligned")
print("  (b) define cosine(0, y) := 0 inside your implementation")
print("Silently emitting nan is not one of them.")

# --- your decision goes here ---

In [ ]:
def cosine_scratch(A, B=None):
    """Pairwise cosine similarity, from scratch in numpy.

        cos(x, y) = (x . y) / (||x||_2 ||y||_2)

    Parameters
    ----------
    A : ndarray (n, d)
        Dense. Convert sparse input with `.toarray()` before calling.
    B : ndarray (m, d), optional
        If None, compare A against itself (m = n).

    Returns
    -------
    ndarray (n, m)
        Entry [i, j] = cos(A[i], B[j]). In [-1, 1] generally; in [0, 1] for TF-IDF,
        since those features are non-negative.

    Notes
    -----
    No Python loop over pairs. The whole thing is two row-normalisations and one
    matrix product — which is also the reason cosine is cheap at scale.
    Decide what a zero row should return (see the cell above).
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit)

# TRAP 1: scipy.spatial.distance.cosine is a DISTANCE, not a similarity.
from scipy.spatial.distance import cosine as scipy_cosine_DISTANCE

a, b = np.array([1.0, 2, 3]), np.array([2.0, 4, 6])   # b = 2a: identical direction
print(f"scipy.spatial.distance.cosine(a, 2a) = {scipy_cosine_DISTANCE(a, b):.3f}"
      "   <- 0 means IDENTICAL, not unrelated")
print(f"the similarity you meant is 1 - that = {1 - scipy_cosine_DISTANCE(a, b):.3f}")
assert np.isclose(scipy_cosine_DISTANCE(a, b), 0.0)

# your implementation vs sklearn's, on a dense slice
sample = X_text[:200].toarray()
mine, ref = cosine_scratch(sample), cosine_similarity(sample)
assert mine.shape == ref.shape, f"shape {mine.shape}, expected {ref.shape}"
assert not np.isnan(mine).any(), "nan in your cosine — go back to the zero-vector cell"
assert np.allclose(mine, ref, atol=1e-10), f"max |diff| = {np.abs(mine - ref).max():.2e}"
# Only rows that HAVE a direction must self-match at 1.0. If you took option (b),
# a zero row self-matches at 0.0 and that is correct, not a bug.
nz = np.linalg.norm(sample, axis=1) > 0
assert np.allclose(np.diag(mine)[nz], 1.0), "a non-zero vector must self-match at 1.0"
print("cosine_scratch agrees with sklearn.cosine_similarity\n")

# STAGE 1 RESULT — sklearn on the full corpus (yours is O(n^2 d) dense and slower)
S_cos = cosine_similarity(X_text)
p_cos = precision_at_k(top_k_neighbours(S_cos, K, higher_is_better=True), y_text)
RESULTS[("cosine", "text")] = (p_cos.mean(), p_cos.std())

print(f"STAGE 1   TF-IDF + cosine   ({X_text.shape[0]} docs in play)")
print(f"  precision@{K} = {p_cos.mean():.4f}   std across queries = {p_cos.std():.4f}")
print(f"  (random baseline would be ~{1 / len(CATEGORIES):.2f})")
print()
print("This number depends on the zero-vector choice you made above — dropping those")
print("rows and keeping them give different answers. The README lists both; check yours.")

---
## Stage 2 — TF-IDF + Euclidean on *unnormalised* vectors

Same features, same task, one change: rank by straight-line distance instead of
angle. Watch what happens.

`‖x‖` for a TF-IDF row grows with document length. So `d(x, y)` is largely
answering *"are these two posts a similar length?"* rather than *"are they about
the same thing?"*. The norm distribution printed below is the evidence — look at
the ratio between the longest and shortest document before you look at the score.

Euclidean is Minkowski at `p = 2`; you'll generalise it in stage 4.

In [ ]:
def euclidean_scratch(A, B=None):
    """Pairwise Euclidean distance, from scratch in numpy.

        d(x, y) = sqrt( SUM_i (x_i - y_i)^2 )

    Parameters
    ----------
    A : ndarray (n, d)   — dense; use .toarray() on sparse input first.
    B : ndarray (m, d), optional — if None, compare A against itself.

    Returns
    -------
    ndarray (n, m) of float, entry [i, j] = d(A[i], B[j]).
        Non-negative, and zero on the diagonal when B is None.

    Notes
    -----
    The textbook expansion ||x-y||^2 = ||x||^2 + ||y||^2 - 2 x.y turns this into
    one matrix product, which is what sklearn does. It also loses precision for
    very close points, where the subtraction cancels. Either route is fine here;
    if your diagonal comes out as small negative numbers before the sqrt, that
    cancellation is exactly what you're looking at.
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit)
import matplotlib.pyplot as plt

sample = X_text[:200].toarray()
mine, ref = euclidean_scratch(sample), euclidean_distances(sample)
assert mine.shape == ref.shape
assert np.allclose(np.diag(mine), 0.0, atol=1e-6), "distance from a point to itself must be 0"
assert (mine >= -1e-12).all(), "negative distance -> you skipped the sqrt or clipped wrong"

# Compare RELATIVE to the scale of the data: these distances run into the hundreds,
# so a fixed atol would be meaningless. The absolute gap is worth looking at though.
adiff = np.abs(mine - ref).max()
rdiff = adiff / ref.max()
print(f"vs sklearn: max |diff| = {adiff:.2e}, relative to the largest distance = {rdiff:.2e}")
assert rdiff < 1e-8, f"relative error {rdiff:.2e} is too large to be rounding"
print("euclidean_scratch agrees with sklearn.euclidean_distances")
print("(If that absolute gap is ~1e-6 you used the ||x||^2+||y||^2-2x.y expansion, and")
print(" you are seeing the cancellation its docstring warned about — it bites hardest")
print(" on near-duplicate documents, where the true distance is almost 0. Direct")
print(" subtraction lands nearer 1e-12. Both are correct; only one is well-conditioned.)\n")

# The norm distribution: the actual explanation for the score below.
norms = np.sqrt(np.asarray(X_text.multiply(X_text).sum(axis=1))).ravel()
q = np.percentile(norms, [0, 25, 50, 75, 100])
print("||x||  min/25%/50%/75%/max: " + "  ".join(f"{v:.2f}" for v in q))
if q[0] > 0:
    print(f"longest:shortest norm ratio = {q[-1] / q[0]:.1f}x")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(norms, bins=60, color="steelblue")
ax[0].set(title="TF-IDF norms (linear)", xlabel="||x||", ylabel="documents")
ax[1].hist(norms[norms > 0], bins=60, color="indianred")
ax[1].set(title="non-zero norms (log x)", xlabel="||x||"); ax[1].set_xscale("log")
plt.tight_layout(); plt.show()

D_raw = euclidean_distances(X_text)
p_euc_raw = precision_at_k(top_k_neighbours(D_raw, K, higher_is_better=False), y_text)
RESULTS[("euclidean-raw", "text")] = (p_euc_raw.mean(), p_euc_raw.std())
print(f"STAGE 2   TF-IDF + Euclidean, unnormalised")
print(f"  precision@{K} = {p_euc_raw.mean():.4f}   std across queries = {p_euc_raw.std():.4f}")
print(f"  stage 1 (cosine) was {RESULTS[('cosine', 'text')][0]:.4f}")

---
## Stage 3 — L2-normalise, then Euclidean. **The key identity.**

This is the stage the whole geometric family hangs on.

Put every vector on the unit sphere (`‖x‖ = 1`). Then expand the squared distance:

> `‖x − y‖² = ‖x‖² + ‖y‖² − 2·x·y = 1 + 1 − 2·cos(x,y) = 2(1 − cos(x,y))`

`d²` is a strictly **decreasing** function of `cos`. A strictly decreasing
transform cannot change a ranking. So on normalised vectors, **cosine and
Euclidean are the same metric** — sorting by one *is* sorting by the other.

Which means stages 1 and 2 were never a contest between two metrics. They were a
contest between normalising and not normalising.

The verification cell checks this three ways and prints loudly. One of the three
is subtler than it looks — **read the stage 3 section of the README after you run
it**, before concluding you have a bug.

In [ ]:
def l2_normalize(X):
    """Scale every row to unit L2 length.

        x  ->  x / ||x||_2        so that afterwards  ||x||_2 == 1

    Parameters
    ----------
    X : sparse matrix or ndarray, shape (n, d)

    Returns
    -------
    Same type and shape as X, every row having unit norm.

    Notes
    -----
    Rows, not columns — axis=1. Keep the sparse type sparse if you can; a dense
    2155 x 5000 float64 copy is ~86 MB and you'll build several matrices from it.
    A row with no direction cannot be given unit length; whatever you decided in
    stage 1 has to hold here too.
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit). This one is deliberately loud.

Xn_text = l2_normalize(X_text)
row_norms = np.sqrt(np.asarray(Xn_text.multiply(Xn_text).sum(axis=1))).ravel() \
    if sp.issparse(Xn_text) else np.linalg.norm(Xn_text, axis=1)
n_zero_rows = int((row_norms == 0).sum())
assert np.allclose(row_norms[row_norms > 0], 1.0), "non-zero rows must come out at unit norm"
assert np.allclose(normalize(X_text).toarray(), Xn_text.toarray() if sp.issparse(Xn_text)
                   else Xn_text, atol=1e-10), "disagrees with sklearn.preprocessing.normalize"
print(f"l2_normalize OK ({n_zero_rows} rows still have norm 0)\n")

cos_n = np.asarray((Xn_text @ Xn_text.T).todense()) if sp.issparse(Xn_text) else Xn_text @ Xn_text.T
D_n = euclidean_distances(Xn_text)
n = D_n.shape[0]

# (1) the algebra
gap = np.abs(D_n ** 2 - 2 * (1 - cos_n)).max()
print(f"[1] max |d^2 - 2(1-cos)|                          = {gap:.3e}")

# (2) the literal request: are the argsort orders equal?
eye = np.eye(n, dtype=bool)
o_cos = np.argsort(np.where(eye, np.inf, -cos_n), axis=1, kind="stable")
o_euc = np.argsort(np.where(eye, np.inf, D_n), axis=1, kind="stable")
print(f"[2] np.array_equal(argsort cosine, argsort euclidean) = {np.array_equal(o_cos, o_euc)}")
print(f"    ties: {(cos_n == 0).mean():.1%} of pairs have cosine exactly 0 (no shared term)")

# (3) the tie-proof version: walk neighbours in euclidean order, watch cosine.
#     If cosine never RISES, the euclidean order is a valid cosine order.
rise = np.diff(np.take_along_axis(cos_n, o_euc[:, :n - 1], axis=1), axis=1).max()
ndiff = int((top_k_neighbours(cos_n, K, True) != top_k_neighbours(D_n, K, False)).any(1).sum())
print(f"[3] max RISE of cosine along the euclidean order   = {rise:.3e}   (0 => same order)")
print(f"    queries whose top-{K} differs: {ndiff} of {n}")

if n_zero_rows:
    print(f"\n!! {n_zero_rows} rows still have norm 0, so [1] and [3] are large. This is NOT a")
    print("   bug in your code and NOT a broken identity. The identity assumes ||x||=||y||=1.")
    print("   A zero vector cannot be put on the sphere: it stays at the origin, distance 1.0")
    print("   from every unit vector, while real neighbours sit near sqrt(2)=1.41. So every")
    print("   empty document becomes everybody's nearest neighbour. Go back to the stage 1")
    print("   zero-vector cell and drop those rows from X_text, y_text AND docs.")
else:
    assert gap < 1e-9 and rise <= 1e-12, "identity broke -> a real bug, see README stage 3"
    print("\n*** SAME RANKING: cosine IS Euclidean on the unit sphere. ***")

p_euc_l2 = precision_at_k(top_k_neighbours(D_n, K, higher_is_better=False), y_text)
RESULTS[("euclidean-l2", "text")] = (p_euc_l2.mean(), p_euc_l2.std())
print(f"\nSTAGE 3   L2-normalised + Euclidean")
print(f"  precision@{K} = {p_euc_l2.mean():.4f}   std = {p_euc_l2.std():.4f}")
print(f"  stage 1 (cosine)          {RESULTS[('cosine', 'text')][0]:.4f}")
print(f"  stage 2 (raw Euclidean)   {RESULTS[('euclidean-raw', 'text')][0]:.4f}")

---
## Stage 4 — Minkowski sweep over p

Minkowski generalises the family with one dial:

> `d_p(x, y) = ( SUM_i |x_i − y_i|^p )^(1/p)`

`p=1` Manhattan, `p=2` Euclidean, `p→∞` Chebyshev (`max_i |x_i − y_i|`).

The interesting question isn't *which p wins*. It's **whether the spread across p
is bigger than the noise you already have**. You have a std across queries from
every stage so far — if moving p changes the mean by less than that, p isn't
doing anything you could rely on.

Two anchors the verification checks for you: at `p=2` your Minkowski must equal
your stage-2 Euclidean, and on L2-normalised rows `p=2` must reproduce stage 3.

**Cost warning.** Pairwise Minkowski at fractional `p` has no BLAS path — it's
O(n²d) scalar work, ~13 s per p at 600×5000, and it scales quadratically. The
sweep runs on a 600-document subsample for that reason. A fully vectorised
from-scratch version materialises an `(n, n, d)` cube — at n=600, d=5000 that is
**14 GB**, so test yours on a slice of 60 rows, not on the corpus.

In [ ]:
def minkowski_scratch(A, B=None, p=2):
    """Pairwise Minkowski distance of order p, from scratch in numpy.

        d_p(x, y) = ( SUM_i |x_i - y_i|^p )^(1/p)          for 1 <= p < inf
        d_inf(x, y) = max_i |x_i - y_i|                    for p = inf

    Parameters
    ----------
    A : ndarray (n, d)
    B : ndarray (m, d), optional — if None, compare A against itself.
    p : float >= 1, or np.inf

    Returns
    -------
    ndarray (n, m) of float.

    Notes
    -----
    p = inf is a separate branch, not a large number: `(x ** np.inf).sum()`
    overflows to inf. Check `np.isinf(p)` first.
    Broadcasting A[:, None, :] - B[None, :, :] gives an (n, m, d) cube. Correct,
    and 14 GB at n=m=600, d=5000. Fine for the 60-row check below; do not point
    it at the corpus.
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit)
from scipy.spatial.distance import cdist

small = Xn_text[:60].toarray() if sp.issparse(Xn_text) else Xn_text[:60]
for p_ in (1, 1.5, 2, 3):
    ref_ = cdist(small, small, "minkowski", p=p_)
    assert np.allclose(minkowski_scratch(small, p=p_), ref_, atol=1e-8), f"mismatch at p={p_}"
assert np.allclose(minkowski_scratch(small, p=np.inf), cdist(small, small, "chebyshev"), atol=1e-8), \
    "p=inf branch is wrong — did you special-case it?"
gap_p2 = np.abs(minkowski_scratch(small, p=2) - euclidean_scratch(small)).max()
assert gap_p2 < 1e-5, f"p=2 must reproduce your own stage-2 Euclidean (gap {gap_p2:.2e})"
print("minkowski_scratch agrees with scipy at p = 1, 1.5, 2, 3, inf")
print(f"p=2 vs your own euclidean_scratch: max |diff| = {gap_p2:.2e}")
print("(~1e-16 means both use the same route; ~1e-8 means one expands and one subtracts")
print(" directly. Same metric, different conditioning — see the stage 2 note.)\n")

# THE SWEEP — scipy on a 600-doc subsample (yours would be 14 GB; see the markdown)
MINK_N = 600
idx = np.random.default_rng(SEED).choice(X_text.shape[0], MINK_N, replace=False)
A_sub = (Xn_text[idx].toarray() if sp.issparse(Xn_text) else Xn_text[idx])
y_sub = y_text[idx]

means, stds, per_p = [], [], {}
for p_ in (1, 1.5, 2, 3, np.inf):
    D_p = cdist(A_sub, A_sub, "chebyshev") if np.isinf(p_) else cdist(A_sub, A_sub, "minkowski", p=p_)
    pr = precision_at_k(top_k_neighbours(D_p, K, higher_is_better=False), y_sub)
    means.append(pr.mean()); stds.append(pr.std()); per_p[p_] = pr
    RESULTS[(f"minkowski p={p_}", "text")] = (pr.mean(), pr.std())
    print(f"  p={str(p_):4s}  precision@{K} = {pr.mean():.4f}   std across queries = {pr.std():.4f}")

# anchor: on unit-norm rows, p=2 must reproduce the cosine ranking (stage 3)
p_anchor = precision_at_k(top_k_neighbours(A_sub @ A_sub.T, K, True), y_sub)
print(f"\n  anchor: cosine on the same {MINK_N} docs = {p_anchor.mean():.4f} vs p=2 "
      f"{per_p[2].mean():.4f}  (diff {abs(p_anchor.mean() - per_p[2].mean()):.4f})")

spread, noise = max(means) - min(means), float(np.mean(stds))
se = noise / np.sqrt(MINK_N)
print(f"\n  spread across p      = {spread:.4f}")
print(f"  mean per-query std   = {noise:.4f}   <- the yardstick you asked for")
print(f"  standard error of a mean = {se:.4f}   <- the stricter, more honest yardstick")
print(f"  -> p is {'REAL' if spread > noise else 'WITHIN NOISE'} against the per-query std")

d_ = per_p[2] - per_p[1.5]
print(f"  paired: p=2 minus p=1.5 = {d_.mean():+.4f} +- {d_.std(ddof=1) / np.sqrt(len(d_)):.4f} (SE)")

---
## Stage 5 — weighted cosine with `w = idf²`

Give each dimension its own importance via a diagonal matrix `W = diag(w)`:

> `cos_W(x, y) = xᵀWy / sqrt( (xᵀWx) · (yᵀWy) )`

Now the claim to verify. Run this on raw **counts** with `w = idf²`. Since W is
diagonal and non-negative,

> `xᵀWy = SUM_i w_i x_i y_i = (x∘idf) · (y∘idf)`

and the same substitution happens in both norms — so weighted cosine on counts is
*exactly* plain cosine on TF-IDF. Weighting the **metric** and weighting the
**features** are the same operation with two different names.

That's worth internalising before set-based measures: a diagonal `W` never buys
you a new metric, only a change of basis. Stage 6 breaks that by making the
weight matrix non-diagonal, which is where it stops being a relabelling.

In [ ]:
# DATA (mine) — raw counts over exactly the TF-IDF vocabulary, plus sklearn's idf.
counts = CountVectorizer(vocabulary=vect.vocabulary_).fit_transform(docs)
idf = vect.idf_
print(f"counts {counts.shape}, idf {idf.shape}, idf range {idf.min():.2f} .. {idf.max():.2f}")
assert counts.shape == X_text.shape, "counts and X_text must line up — did you drop rows from docs too?"

In [ ]:
def weighted_cosine(X, w):
    """Cosine under a diagonal weight matrix W = diag(w).

        cos_W(x, y) = (x^T W y) / sqrt( (x^T W x) * (y^T W y) )

    Parameters
    ----------
    X : sparse matrix or ndarray (n, d)
    w : ndarray (d,) of non-negative weights.

    Returns
    -------
    ndarray (n, n), entry [i, j] = cos_W(X[i], X[j]); w = ones reduces to plain cosine.

    Notes
    -----
    You can form this directly, or notice that for a non-negative diagonal W,
    x^T W y = (sqrt(W) x) . (sqrt(W) y) exactly — which turns the whole thing into
    a plain cosine on rescaled features and costs one elementwise multiply.
    Both routes must give the same matrix; that equivalence IS the lesson.
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit)

# TF-IDF really is counts * idf (sublinear_tf=False, norm=None), so the premise holds.
assert np.allclose(counts[:50].multiply(idf).toarray(), X_text[:50].toarray(), atol=1e-10), \
    "tfidf != counts * idf — check the vectorizer settings"

# sanity: w = ones must collapse to plain cosine
sub = counts[:150]
assert np.allclose(weighted_cosine(sub, np.ones(counts.shape[1])),
                   cosine_similarity(sub), atol=1e-8), "w=1 must give plain cosine"

cw = weighted_cosine(counts, idf ** 2)
c1 = cosine_similarity(X_text)
print(f"max |weighted_cosine(counts, idf^2) - cosine(tfidf)| = {np.abs(cw - c1).max():.3e}")
assert np.allclose(cw, c1, atol=1e-8), "the identity failed — is w idf SQUARED?"

p_w = precision_at_k(top_k_neighbours(cw, K, higher_is_better=True), y_text)
RESULTS[("weighted-cosine", "text")] = (p_w.mean(), p_w.std())
print(f"\nSTAGE 5   weighted cosine, w = idf^2")
print(f"  precision@{K} = {p_w.mean():.4f}   std = {p_w.std():.4f}")
print(f"  stage 1 was  {RESULTS[('cosine', 'text')][0]:.4f}  <- must match to the 4th decimal")

---
## Stage 6 — soft cosine (a full, non-diagonal `S`)

Plain cosine says *car* and *automobile* share nothing: different dimensions, dot
product zero. Soft cosine fixes that with a term-similarity matrix `S`:

> `soft_cos(x, y) = xᵀSy / sqrt( (xᵀSx) · (yᵀSy) )`

Same shape as stage 5, one crucial difference: `S` is **not diagonal**, so it
cannot be absorbed into the features by rescaling. `S = I` gives plain cosine back.

**`S` must be positive semi-definite.** If it isn't, `xᵀSx` can go negative and
the denominator takes the square root of a negative number. The toy half below
exists so you can see this directly: build `S` as a Gram matrix `E Eᵀ` — PSD by
construction — and read its eigenvalues.

We do the toy first, on five words, where you can print `S` and check it by eye.

In [ ]:
# DATA (mine) — a five-word vocabulary with hand-made 3-d "embeddings".
toy_vocab = ["car", "automobile", "vehicle", "banana", "fruit"]
E_toy = np.array([[1.00, 0.00, 0.10],
                  [0.95, 0.05, 0.10],
                  [0.90, 0.10, 0.20],
                  [0.00, 1.00, 0.00],
                  [0.05, 0.95, 0.10]])
E_toy /= np.linalg.norm(E_toy, axis=1, keepdims=True)

# Two documents with ZERO lexical overlap: one says "car", the other "automobile".
doc_car = np.array([1.0, 0, 0, 0, 0])
doc_auto = np.array([0, 1.0, 0, 0, 0])
print("vocabulary:", toy_vocab)
print("build S with your own cosine_scratch(E_toy) — it is a Gram matrix, hence PSD.")

In [ ]:
def soft_cosine(X, S):
    """Soft cosine similarity under a term-similarity matrix S.

        soft_cos(x, y) = (x^T S y) / sqrt( (x^T S x) * (y^T S y) )

    Parameters
    ----------
    X : sparse matrix or ndarray (n, d) — documents in rows.
    S : sparse matrix or ndarray (d, d) — symmetric, PSD, unit diagonal.

    Returns
    -------
    ndarray (n, n), entry [i, j] = soft_cos(X[i], X[j]). S = I reduces to cosine.

    Notes
    -----
    Do NOT loop over pairs. Form M = X S X^T once; then the numerators are M
    itself and the two norms are sqrt(diag(M)) — every quantity you need is in
    that one product.
    If any diag(M) <= 0, S was not PSD. Don't silently clamp it: notice it.
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit): toy vocabulary, S small enough to read.

S_toy = cosine_scratch(E_toy)            # your stage-1 function, reused
print("S =")
for w_, row in zip(toy_vocab, S_toy):
    print(f"  {w_:>11s} " + " ".join(f"{v:6.3f}" for v in row))

ev = np.linalg.eigvalsh(S_toy)
print(f"\neigenvalues = {np.round(ev, 4)}")
print(f"min eigenvalue = {ev.min():.2e}  ->  PSD = {ev.min() >= -1e-9}")
assert np.allclose(S_toy, S_toy.T), "S must be symmetric"
assert np.allclose(np.diag(S_toy), 1.0), "unit diagonal: every term matches itself"
assert ev.min() >= -1e-9, "a Gram matrix should be PSD — check cosine_scratch"

X_toy = np.vstack([doc_car, doc_auto])
plain = cosine_scratch(X_toy)[0, 1]
soft = soft_cosine(X_toy, S_toy)[0, 1]
print(f"\nplain cosine('car', 'automobile') = {plain:.3f}   <- no shared term, exactly 0")
print(f"soft  cosine('car', 'automobile') = {soft:.3f}   <- S carries the synonymy")
assert np.isclose(plain, 0.0), "these documents share no term"
assert soft > 0.9, "soft cosine should recover the near-synonymy"
assert np.allclose(soft_cosine(X_toy, np.eye(5)), cosine_scratch(X_toy), atol=1e-10), \
    "S = I must reduce exactly to plain cosine"
print("\ntoy soft cosine OK — including S = I collapsing to stage 1")

### Stage 6b — soft cosine on the real corpus

Now `S` is 5000×5000. Two things change and both matter.

**Where the term vectors come from.** The brief asks for gensim's
`SparseTermSimilarityMatrix` with `glove-wiki-gigaword-100`. **gensim will not
install on this machine** — Python 3.14, no wheel, and the source build fails on
missing `Python.h`. The loader below tries gensim first and falls back to LSA term
vectors derived from this corpus (a `TruncatedSVD` of the term–document matrix:
same distributional idea, computed locally). See the README for how to get the
real GloVe path if you want it. The maths you write is identical either way.

**PSD stops being free.** A dense 5000×5000 float64 `S` is **200 MB**, so `S` is
normally truncated — keep each term's top-k neighbours, drop the rest. That is
what gensim does too, and it is exactly what can destroy the PSD property the toy
example handed you. Your `soft_cosine` should notice, not paper over it.

Then the payoff question: **which** queries improved? The standard story is that
soft cosine rescues short documents with little lexical overlap. The last cell
measures that claim rather than repeating it.

In [ ]:
# DATA (mine) — term vectors: gensim/GloVe if available, else LSA from this corpus.
def load_term_vectors(vectorizer, X_norm):
    terms = vectorizer.get_feature_names_out()
    try:
        import gensim.downloader as api
        kv = api.load("glove-wiki-gigaword-100")
        E = np.array([kv[t] if t in kv else np.zeros(kv.vector_size) for t in terms])
        return E, "GloVe glove-wiki-gigaword-100 (gensim)"
    except Exception as exc:
        from sklearn.decomposition import TruncatedSVD
        E = TruncatedSVD(100, random_state=SEED).fit_transform(X_norm.T)
        return E, f"LSA term vectors ({type(exc).__name__}: gensim unavailable)"

E_terms, E_source = load_term_vectors(vect, Xn_text)
print(f"term vectors: {E_terms.shape} from {E_source}")

In [ ]:
def build_S(E, topk=5, thresh=0.5, alpha=0.3):
    """Term-similarity matrix from term embeddings, truncated to top-k per term.

        S[i, j] = 1                        if i == j
                = alpha * cos(E_i, E_j)    if j is among i's topk AND cos > thresh
                = 0                        otherwise
        then symmetrise:  S <- (S + S^T) / 2  with the unit diagonal restored.

    Parameters
    ----------
    E : ndarray (T, k) — one embedding row per vocabulary term (T = 5000 here).
    topk : int   — neighbours kept per term.
    thresh : float — minimum cosine to keep an off-diagonal entry.
    alpha : float — global damping on off-diagonals.

    Returns
    -------
    (T, T) matrix, symmetric with unit diagonal. Sparse strongly preferred:
    dense float64 at T=5000 is 200 MB, and you still have to multiply by it.

    Notes
    -----
    Do not build the full T x T cosine matrix in one go — block over rows
    (say 500 at a time) and keep only what survives topk/thresh.
    Exclude each term from its own top-k before you take them, or every term's
    best neighbour is itself.
    `alpha` and `thresh` are what keep S diagonally dominant, which is what keeps
    it PSD after truncation. Turn alpha up and watch that guarantee die.
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit)
S_full = build_S(E_terms)
T = E_terms.shape[0]
assert S_full.shape == (T, T), f"S must be {(T, T)}, got {S_full.shape}"
nnz = S_full.nnz if sp.issparse(S_full) else int((S_full != 0).sum())
diag = np.asarray(S_full.diagonal()).ravel() if sp.issparse(S_full) else np.diag(S_full)
assert np.allclose(diag, 1.0), "unit diagonal required"
assert abs(S_full - S_full.T).max() < 1e-9, "S must be symmetric"
print(f"S: {T}x{T}, {nnz} non-zeros ({nnz / T:.1f} per term), "
      f"{'sparse' if sp.issparse(S_full) else 'dense'}")

sim_soft = soft_cosine(Xn_text, S_full)
M_diag = np.diag(np.asarray((Xn_text @ S_full @ Xn_text.T).todense())
                 if sp.issparse(Xn_text) else Xn_text @ S_full @ Xn_text.T)
n_neg = int((M_diag <= 0).sum())
if n_neg:
    print(f"!! {n_neg} documents have x^T S x <= 0 -> your truncated S is NOT PSD.")
    print("   Expected consequence of top-k truncation. Lower alpha or raise thresh.")
else:
    print("all x^T S x > 0 — truncation left S usable (diagonal dominance saved you)")

p_soft = precision_at_k(top_k_neighbours(sim_soft, K, higher_is_better=True), y_text)
RESULTS[("soft-cosine", "text")] = (p_soft.mean(), p_soft.std())
base = p_cos  # stage 1, per query
d_ = p_soft - base
print(f"\nSTAGE 6   soft cosine")
print(f"  precision@{K} = {p_soft.mean():.4f}   std = {p_soft.std():.4f}")
print(f"  stage 1 was   {base.mean():.4f}")
print(f"  paired diff   {d_.mean():+.4f} +- {d_.std(ddof=1) / np.sqrt(len(d_)):.4f} (SE)  "
      f"-> {'REAL' if abs(d_.mean()) > 2 * d_.std(ddof=1) / np.sqrt(len(d_)) else 'WITHIN NOISE'}")
print(f"  improved {int((d_ > 0).sum())}, hurt {int((d_ < 0).sum())}, unchanged {int((d_ == 0).sum())}")

# Does it really rescue SHORT, low-overlap documents? Measure, don't assume.
n_terms_doc = np.asarray((Xn_text > 0).sum(1)).ravel()
n_words_doc = np.array([len(t.split()) for t in docs])
print(f"\n  corr(gain, distinct terms) = {np.corrcoef(d_, n_terms_doc)[0, 1]:+.3f}")
print(f"  corr(gain, word count)     = {np.corrcoef(d_, n_words_doc)[0, 1]:+.3f}")
print("  most-improved queries — judge the 'short and low-overlap' claim yourself:")
for i in np.argsort(-d_)[:5]:
    print(f"    #{i:5d}  {base[i]:.1f} -> {p_soft[i]:.1f}   {n_words_doc[i]:5d} words, "
          f"{n_terms_doc[i]:3d} terms   {docs[i].strip()[:44]!r}")

---
## Stage 7 — wine: cosine vs Euclidean vs **Mahalanobis**

New dataset, opposite intuitions. 178 wines, 13 numeric chemical measurements,
3 cultivars. Standardise first — raw `proline` is ~1000 and `magnesium` ~100, so
without z-scoring one column owns every distance and the other twelve are
rounding error.

### What the inverse covariance matrix is doing

Mahalanobis distance is

> `d(x, y)² = (x − y)ᵀ C⁻¹ (x − y)`

Euclidean is the **special case `C = I`** — an implicit claim that every feature
is independent and equally scaled. `C⁻¹` withdraws that claim in two ways:

1. **Scale.** Divides each direction by its standard deviation, so a step of "1"
   means "1 std of the data", not "1 raw unit". Z-scoring already did this for
   individual features, so this part is mostly spent here.
2. **Correlation** — the part z-scoring *cannot* do. If two features move
   together, they carry largely the same fact. Euclidean counts that fact twice.
   `C⁻¹` decorrelates ("whitens") the space so it's counted once.

Geometrically: Euclidean measures distance in spheres, Mahalanobis in ellipsoids
shaped by the data's own spread — distance measured in *standard deviations along
each principal axis*, not raw units.

So `C⁻¹` should help whenever correlated features exist. The correlation matrix is
printed below so you can check whether they actually do here — **and then whether
Mahalanobis wins because of it.** Those are two different questions, and this
dataset is interesting precisely because the answers differ. Look at *what* the
correlated features are before you decide the result is a bug.

In [ ]:
# DATA + the correlation matrix (mine)
wine = load_wine()
Z_wine = StandardScaler().fit_transform(wine.data)
y_wine = wine.target
feat = list(wine.feature_names)
print(f"{Z_wine.shape[0]} wines x {Z_wine.shape[1]} features, {len(set(y_wine))} cultivars\n")

C_corr = np.corrcoef(Z_wine, rowvar=False)   # on z-scored data, covariance == correlation
print("feature correlation matrix:")
print("        " + " ".join(f"{n[:5]:>6s}" for n in feat))
for n_, row in zip(feat, C_corr):
    print(f"  {n_[:6]:>6s} " + " ".join(f"{v:6.2f}" for v in row))

iu = np.triu_indices_from(C_corr, 1)
top3 = np.argsort(-np.abs(C_corr[iu]))[:3]
print("\nstrongest correlated pairs:")
for t in top3:
    print(f"  {feat[iu[0][t]]:>22s} ~ {feat[iu[1][t]]:<22s} {C_corr[iu][t]:+.2f}")
ev_c = np.linalg.eigvalsh(np.cov(Z_wine, rowvar=False))
print(f"\ncovariance eigenvalues {ev_c.min():.3f} .. {ev_c.max():.3f}, "
      f"condition number {ev_c.max() / ev_c.min():.1f}")
print(f"-> whitening stretches the weakest direction {np.sqrt(ev_c.max() / ev_c.min()):.1f}x "
      "relative to the strongest.")

In [ ]:
# TRAP 3, planted on purpose — standardise INSIDE the fold, not before it.
#
# Fitting the scaler on all the data before splitting lets the test rows help
# decide where "mean zero" is. That is leakage: mild for a scaler, severe for
# Mahalanobis, whose C^-1 is itself ESTIMATED FROM THE DATA. Fit that on
# everything and you leak the test set's entire geometry, not just its mean.
from sklearn.model_selection import train_test_split

tr, te = train_test_split(np.arange(len(y_wine)), test_size=0.3,
                          random_state=SEED, stratify=y_wine)
sc_all = StandardScaler().fit(wine.data)            # WRONG: saw the test rows
sc_train = StandardScaler().fit(wine.data[tr])      # RIGHT: train only
print(f"the two scalers' centres differ by up to {np.abs(sc_all.mean_ - sc_train.mean_).max():.2f} raw units")
print(f"test rows via the ALL-data scaler   -> mean {sc_all.transform(wine.data[te]).mean():+.4f}")
print(f"test rows via the TRAIN-only scaler -> mean {sc_train.transform(wine.data[te]).mean():+.4f}")
print("the first sits closer to 0 precisely because those rows helped define where 0 is.\n")

C_all = np.cov(StandardScaler().fit_transform(wine.data), rowvar=False)
C_tr = np.cov(StandardScaler().fit(wine.data[tr]).transform(wine.data[tr]), rowvar=False)
print(f"and the covariance itself differs by up to {np.abs(C_all - C_tr).max():.3f} —")
print("that is the quantity Mahalanobis inverts. Retrieval over a whole dataset (what")
print("we do below) has no train/test split, so nothing here is leaking — but the moment")
print("you put Mahalanobis in a CV loop, C^-1 must be estimated inside each fold.")

In [ ]:
def mahalanobis_scratch(A, VI, B=None):
    """Pairwise Mahalanobis distance.

        d(x, y) = sqrt( (x - y)^T VI (x - y) )        where VI = C^-1

    Parameters
    ----------
    A : ndarray (n, d)
    VI : ndarray (d, d) — the INVERSE covariance matrix (already inverted).
    B : ndarray (m, d), optional — if None, compare A against itself.

    Returns
    -------
    ndarray (n, m) of float, zero on the diagonal when B is None.

    Notes
    -----
    VI = I must give plain Euclidean back — a free check on your implementation.
    Tiny negative values can appear under the sqrt from float error; clip at 0.
    Worth knowing (don't implement it, just notice): Mahalanobis is exactly
    Euclidean after whitening. Factor VI = L L^T and the distance becomes
    ||L^T x - L^T y||_2. Every metric in this notebook has now turned out to be
    Euclidean in some transformed space.
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit)
VI_wine = np.linalg.inv(np.cov(Z_wine, rowvar=False))

assert np.allclose(mahalanobis_scratch(Z_wine[:40], np.eye(13)),
                   euclidean_scratch(Z_wine[:40]), atol=1e-8), \
    "VI = I must reduce to plain Euclidean"
assert np.allclose(mahalanobis_scratch(Z_wine[:40], VI_wine),
                   cdist(Z_wine[:40], Z_wine[:40], "mahalanobis", VI=VI_wine), atol=1e-8), \
    "disagrees with scipy cdist"
print("mahalanobis_scratch agrees with scipy, and reduces to Euclidean at VI = I\n")

res7 = {}
res7["euclidean RAW (unscaled)"] = precision_at_k(
    top_k_neighbours(euclidean_distances(wine.data), K, False), y_wine)
res7["cosine"] = precision_at_k(top_k_neighbours(cosine_similarity(Z_wine), K, True), y_wine)
res7["euclidean"] = precision_at_k(top_k_neighbours(euclidean_distances(Z_wine), K, False), y_wine)
res7["mahalanobis"] = precision_at_k(
    top_k_neighbours(mahalanobis_scratch(Z_wine, VI_wine), K, False), y_wine)

print(f"STAGE 7   wine, precision@{K}")
for name, pv in res7.items():
    print(f"  {name:26s} {pv.mean():.4f}   std {pv.std():.4f}")
    if "RAW" not in name:
        RESULTS[(name, "wine")] = (pv.mean(), pv.std())

print("\npaired comparisons (same queries, so compare per query):")
for a_, b_ in [("cosine", "euclidean"), ("euclidean", "mahalanobis")]:
    d_ = res7[a_] - res7[b_]
    se_ = d_.std(ddof=1) / np.sqrt(len(d_))
    print(f"  {a_:12s} - {b_:12s} = {d_.mean():+.4f} +- {se_:.4f}  ->  "
          f"{'REAL' if abs(d_.mean()) > 2 * se_ else 'WITHIN NOISE'}")
print("\nBefore calling any of this a bug, re-read the correlation matrix above and ask")
print("what the correlated block actually IS in this dataset. README stage 7 when you're done.")

---
## Stage 8 — the four distance → similarity conversions

Retrieval ranks; most downstream code wants a bounded *score*. Four standard ways
to turn a distance `d ≥ 0` into a similarity:

| | formula | at d=0 | as d→∞ |
|---|---|---|---|
| reciprocal | `1 / (1 + d)` | 1 | → 0 |
| Gaussian / RBF | `exp(−γd²)` | 1 | → 0 fast |
| exponential / Laplacian | `exp(−γd)` | 1 | → 0 |
| bounded linear | `1 − d/d_max` | 1 | 0 at `d_max` |

All four are strictly **decreasing** in `d`. You proved the consequence in stage
3: a strictly decreasing transform cannot change a ranking. So all four give
**identical retrieval order** — the choice only changes the *values*.

Which is not the same as saying it doesn't matter. Compare `exp(−γd²)` and
`1 − d/d_max` at `d = 3`. If a threshold, a kernel, or a downstream loss consumes
these numbers, that difference is enormous. If you only ever `argsort`, it's
invisible.

In [ ]:
def sim_reciprocal(d):
    """s = 1 / (1 + d).  d: ndarray, any shape. Returns same shape, in (0, 1]."""
    raise NotImplementedError


def sim_gaussian(d, gamma=1.0):
    """s = exp(-gamma * d**2).  d: ndarray, any shape. Returns same shape, in (0, 1]."""
    raise NotImplementedError


def sim_exponential(d, gamma=1.0):
    """s = exp(-gamma * d).  d: ndarray, any shape. Returns same shape, in (0, 1]."""
    raise NotImplementedError


def sim_bounded(d, d_max=None):
    """s = 1 - d / d_max, the bounded-linear conversion.

    d : ndarray, any shape.
    d_max : float or None — if None, use d.max(). Returns same shape, in [0, 1].

    Unlike the other three this one hits exactly 0 at d_max and goes NEGATIVE
    beyond it, so d_max has to be a real upper bound on the distances you feed it.
    """
    raise NotImplementedError

In [ ]:
# VERIFICATION (mine — run, don't edit)
d_grid = np.array([0.0, 0.5, 1.0, 2.0, 3.0, 5.0])
conv = {"reciprocal  1/(1+d)": sim_reciprocal(d_grid),
        "gaussian    exp(-g d^2)": sim_gaussian(d_grid),
        "exponential exp(-g d)": sim_exponential(d_grid),
        "bounded     1 - d/dmax": sim_bounded(d_grid)}

print("  d" + " " * 24 + " ".join(f"{v:8.2f}" for v in d_grid))
for name, s_ in conv.items():
    print(f"  {name:24s}" + " ".join(f"{v:8.4f}" for v in s_))

for name, s_ in conv.items():
    assert s_.shape == d_grid.shape, f"{name}: shape changed"
    assert np.isclose(s_[0], 1.0), f"{name}: d=0 must give similarity 1"
    assert np.all(np.diff(s_) < 0), f"{name}: must be strictly decreasing in d"

orders = [np.argsort(-s_, kind="stable") for s_ in conv.values()]
same = all(np.array_equal(orders[0], o) for o in orders[1:])
print(f"\n  identical argsort across all four = {same}")
assert same, "a monotone decreasing transform cannot reorder — check your formulas"

# and on a real distance matrix, not just a grid
D_small = euclidean_distances(Z_wine)
ref_order = top_k_neighbours(D_small, K, higher_is_better=False)
for name, fn in [("reciprocal", sim_reciprocal), ("gaussian", sim_gaussian),
                 ("exponential", sim_exponential),
                 ("bounded", lambda x: sim_bounded(x, D_small.max()))]:
    assert np.array_equal(top_k_neighbours(fn(D_small), K, higher_is_better=True), ref_order), \
        f"{name} changed the neighbour order on real data"
print("  all four reproduce the Euclidean neighbour order on the wine distance matrix")
print("\n  -> the conversion changes calibration, never retrieval order.")
print("     Compare gaussian vs bounded at d=3 to see how much calibration can differ.")

---
## Summary — metric × dataset × precision@10

Everything the notebook computed, in one table. Run the stages above first; this
cell only formats what they left in `RESULTS`.

When you read it, the comparison worth making is not "which metric won" but **how
much any metric choice moved the number, versus how much the preprocessing
decisions moved it**.

In [ ]:
# (mine — formatting only)
print(f"{'metric':<24}{'dataset':<9}{'precision@' + str(K):>13}{'std':>9}")
print("-" * 55)
for (m_, ds_), (mu_, sd_) in sorted(RESULTS.items(), key=lambda kv: (kv[0][1], -kv[1][0])):
    print(f"{m_:<24}{ds_:<9}{mu_:>13.4f}{sd_:>9.4f}")
print("-" * 55)
print(f"{len(RESULTS)} entries. Missing rows just mean you haven't run that stage yet.")